In [ ]:
import json
import pandas as pd

from emu_renewal.constants import DATA_PATH
from emu_renewal.inputs import get_oxcgrt_data
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats, find_oc_missing_vacc
from emu_renewal.utils import get_country_name

In [1]:
from emu_renewal.inputs import get_oxcgrt

In [6]:
get_oxcgrt("SGP", "custom").index[-1]

Timestamp('2022-12-31 00:00:00')

In [ ]:
analysis = "oxcgrt"

countries = get_oxcgrt_data()["CountryCode"].unique()

In [ ]:
filename = "owid/share-of-people-who-completed-the-initial-covid-19-vaccination-protocol.csv"
data = pd.read_csv(DATA_PATH / filename, index_col="Day")

In [ ]:
summary = pd.DataFrame(index=countries)
death_data, case_data = gather_who_data(countries)
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_reps, case_reps = find_nans_repeats(death_data, case_data, summary)
no_oc_vacc = find_oc_missing_vacc(countries, summary)

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_reps + case_reps + death_outliers + case_outliers + no_oc_vacc)
included = [c for c in countries if c not in excluded]
add_bool_row_to_table(summary, included, "Included")

In [ ]:
summary.index = summary.index.map(get_country_name)
summary

In [ ]:
json.dump(included, open(DATA_PATH / f"config/{analysis}_included.json", "w"))